# Mini-LLaMA From Scratch: Training a Modern LLM with PyTorch

**What you'll learn:**
- Modern transformer architecture (LLaMA 3 / Mistral style)
- RoPE (Rotary Position Embeddings)
- RMSNorm (instead of LayerNorm)
- SwiGLU activation (instead of ReLU/GELU)
- Grouped Query Attention (GQA)
- Character-level tokenization → BPE with tiktoken
- Training loop with mixed precision (AMP)
- Loss visualization and text generation

**Hardware:** NVIDIA 3080 Ti (12GB VRAM) — all configs are tuned for this GPU.

**Disk space needed:** ~2-3 GB total (model + data + checkpoints)

## 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import math
import os
import urllib.request

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Data: Download Shakespeare

We'll use the complete works of Shakespeare (~1MB). Small enough to iterate fast,
large enough to learn real patterns.

In [ ]:
data_path = 'data/shakespeare.txt'
os.makedirs('data', exist_ok=True)

if not os.path.exists(data_path):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    print('Downloading Shakespeare...')
    urllib.request.urlretrieve(url, data_path)

with open(data_path, 'r') as f:
    text = f.read()

print(f'Dataset size: {len(text):,} characters')
print(f'First 200 chars:\n{text[:200]}')

## 2. Tokenizer: Character-Level

We start with a character-level tokenizer for simplicity. This lets us focus on the
model architecture without worrying about BPE complexity.

**Why character-level?**
- Vocabulary is tiny (65 chars) → small embedding table
- No external dependencies
- Easy to debug and understand

**In production**, you'd use BPE (tiktoken, sentencepiece) for much better performance.

In [ ]:
# Build character-level tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'Vocabulary size: {vocab_size}')
print(f'Characters: {"".join(chars)}')

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(tokens):
    return ''.join([itos[t] for t in tokens])

# Encode the full dataset
data = torch.tensor(encode(text), dtype=torch.long)
print(f'Encoded shape: {data.shape}')

# Train/val split (90/10)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f'Train: {len(train_data):,} tokens | Val: {len(val_data):,} tokens')

## 3. Data Loader

We create random batches of sequences. Each input sequence of length `seq_len` has
a target that is the same sequence shifted by one position.

In [ ]:
def get_batch(split, batch_size, seq_len):
    """Get a random batch of training data."""
    d = train_data if split == 'train' else val_data
    # Random starting indices
    ix = torch.randint(len(d) - seq_len, (batch_size,))
    x = torch.stack([d[i:i+seq_len] for i in ix])
    y = torch.stack([d[i+1:i+seq_len+1] for i in ix])
    return x.to(device), y.to(device)

# Test it
xb, yb = get_batch('train', batch_size=4, seq_len=32)
print(f'Input shape:  {xb.shape}')   # (batch, seq_len)
print(f'Target shape: {yb.shape}')   # (batch, seq_len)
print(f'\nExample input:  "{decode(xb[0].tolist())}"')
print(f'Example target: "{decode(yb[0].tolist())}"')

## 4. Model Architecture: Mini-LLaMA

Now the core of the project. We build a **LLaMA-style transformer** with these
modern components:

| Component | Classic Transformer | Our Mini-LLaMA |
|-----------|--------------------|-----------------|
| Normalization | LayerNorm (post) | **RMSNorm (pre)** |
| Position encoding | Sinusoidal / learned | **RoPE** |
| Activation | ReLU / GELU | **SwiGLU** |
| Attention | Multi-Head | **Grouped Query Attention** |
| Bias | Yes | **No bias** |

### 4.1 RMSNorm

Simpler and faster than LayerNorm. Only rescales by RMS — no mean subtraction, no bias.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, 2019).
    
    Unlike LayerNorm, RMSNorm:
    - Does NOT subtract the mean (no re-centering)
    - Only rescales by the RMS of activations
    - Has no bias term
    - Is ~10-15% faster than LayerNorm
    
    Used in: LLaMA 1/2/3, Mistral, Gemma
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # learnable scale
    
    def forward(self, x):
        # x shape: (batch, seq_len, dim)
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight

### 4.2 Rotary Position Embeddings (RoPE)

RoPE encodes position by **rotating** query and key vectors. This gives the model
a sense of relative position without adding position embeddings.

**Key insight:** The dot product between rotated Q and K depends on their *relative*
position, not absolute. This generalizes better to longer sequences.

Used in: LLaMA, Mistral, GPT-NeoX, PaLM, Gemma

In [ ]:
def precompute_rope_frequencies(dim, max_seq_len, theta=10000.0):
    """Precompute the complex exponentials for RoPE.
    
    Args:
        dim: head dimension (must be even)
        max_seq_len: maximum sequence length to support
        theta: base frequency (10000 is standard)
    
    Returns:
        freqs_cis: complex tensor of shape (max_seq_len, dim//2)
    """
    # Frequency for each dimension pair
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    # Position indices
    t = torch.arange(max_seq_len)
    # Outer product: (seq_len, dim//2)
    freqs = torch.outer(t, freqs)
    # Convert to complex exponentials: e^(i*theta) = cos(theta) + i*sin(theta)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


def apply_rope(x, freqs_cis):
    """Apply rotary embeddings to queries or keys.
    
    Args:
        x: tensor of shape (batch, seq_len, n_heads, head_dim)
        freqs_cis: precomputed frequencies (seq_len, head_dim//2)
    """
    # Reshape x into pairs of consecutive dims and view as complex
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    # Reshape freqs for broadcasting: (1, seq_len, 1, head_dim//2)
    freqs = freqs_cis.unsqueeze(0).unsqueeze(2)
    # Multiply (rotate) and convert back to real
    x_rotated = torch.view_as_real(x_complex * freqs).flatten(-2)
    return x_rotated.type_as(x)


# Visualize RoPE frequencies
demo_freqs = precompute_rope_frequencies(dim=64, max_seq_len=256)
plt.figure(figsize=(10, 4))
plt.imshow(demo_freqs.real.numpy().T, aspect='auto', cmap='RdBu')
plt.xlabel('Position')
plt.ylabel('Dimension pair')
plt.title('RoPE: Cosine components of rotation frequencies')
plt.colorbar()
plt.tight_layout()
plt.show()

### 4.3 Grouped Query Attention (GQA)

Standard multi-head attention uses separate K, V projections for each head.
GQA **shares** K and V across groups of heads, reducing memory and compute.

- **MHA:** n_heads Q, n_heads K, n_heads V (standard)
- **GQA:** n_heads Q, n_kv_heads K, n_kv_heads V (LLaMA 3 uses this)
- **MQA:** n_heads Q, 1 K, 1 V (extreme sharing)

Used in: LLaMA 2 (70B), LLaMA 3, Mistral, Gemini

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Grouped Query Attention with RoPE.
    
    Args:
        dim: model dimension
        n_heads: number of query heads
        n_kv_heads: number of key/value heads (< n_heads for GQA)
    """
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads  # how many Q heads share each KV head
        self.head_dim = dim // n_heads
        
        # Q gets full heads, K and V get fewer
        self.wq = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_heads * self.head_dim, dim, bias=False)
    
    def forward(self, x, freqs_cis):
        B, T, C = x.shape
        
        # Project to Q, K, V
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)
        
        # Apply RoPE to Q and K (not V!)
        q = apply_rope(q, freqs_cis)
        k = apply_rope(k, freqs_cis)
        
        # Repeat K, V heads to match Q heads for GQA
        if self.n_rep > 1:
            k = k.unsqueeze(3).expand(B, T, self.n_kv_heads, self.n_rep, self.head_dim)
            k = k.reshape(B, T, self.n_heads, self.head_dim)
            v = v.unsqueeze(3).expand(B, T, self.n_kv_heads, self.n_rep, self.head_dim)
            v = v.reshape(B, T, self.n_heads, self.head_dim)
        
        # Transpose for attention: (B, n_heads, T, head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # Scaled dot-product attention with causal mask
        # PyTorch 2.0+ has a fused kernel for this — much faster!
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        
        # Reshape back: (B, T, dim)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)

### 4.4 SwiGLU Feed-Forward Network

SwiGLU replaces the standard FFN (Linear → ReLU → Linear) with a gated version
that uses the Swish activation. It consistently outperforms ReLU and GELU.

```
Standard FFN:  Linear(dim, 4*dim) → ReLU → Linear(4*dim, dim)
SwiGLU FFN:    Swish(Linear₁(x)) * Linear₂(x) → Linear₃(out)
```

The hidden dim is typically `(4 * dim * 2/3)` rounded to a multiple of 256.

In [ ]:
class SwiGLU_FFN(nn.Module):
    """SwiGLU Feed-Forward Network (Shazeer, 2020).
    
    Uses gated activation: output = Swish(W1·x) * (W2·x), then project down.
    """
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            # LLaMA convention: 4 * dim * 2/3, rounded to multiple of 256
            hidden_dim = int(2 * (4 * dim) / 3)
            hidden_dim = 256 * ((hidden_dim + 255) // 256)
        
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)  # gate projection
        self.w2 = nn.Linear(dim, hidden_dim, bias=False)   # up projection  
        self.w3 = nn.Linear(hidden_dim, dim, bias=False)   # down projection
    
    def forward(self, x):
        # SwiGLU: Swish(gate) * up, then project down
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

### 4.5 Transformer Block

Each block follows the **pre-norm** pattern (norm before attention/FFN, not after).
This is more stable for training deep networks.

```
x → RMSNorm → Attention → + residual → RMSNorm → FFN → + residual
```

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        self.attention = GroupedQueryAttention(dim, n_heads, n_kv_heads)
        self.ffn = SwiGLU_FFN(dim)
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
    
    def forward(self, x, freqs_cis):
        # Pre-norm + residual connections
        x = x + self.attention(self.norm1(x), freqs_cis)
        x = x + self.ffn(self.norm2(x))
        return x

### 4.6 Full Mini-LLaMA Model

In [ ]:
class MiniLLaMA(nn.Module):
    """A miniature LLaMA-style language model.
    
    Architecture matches LLaMA 3 design:
    - RoPE position encoding
    - RMSNorm (pre-norm)
    - SwiGLU FFN
    - Grouped Query Attention
    - No bias in any linear layers
    """
    def __init__(self, vocab_size, dim, n_layers, n_heads, n_kv_heads, max_seq_len):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        
        # Token embedding (no position embedding — RoPE handles it)
        self.tok_emb = nn.Embedding(vocab_size, dim)
        
        # Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(dim, n_heads, n_kv_heads)
            for _ in range(n_layers)
        ])
        
        # Final norm + output projection
        self.norm = RMSNorm(dim)
        self.output = nn.Linear(dim, vocab_size, bias=False)
        
        # Weight tying: share embedding and output weights
        self.tok_emb.weight = self.output.weight
        
        # Precompute RoPE frequencies
        head_dim = dim // n_heads
        self.register_buffer(
            'freqs_cis',
            precompute_rope_frequencies(head_dim, max_seq_len),
            persistent=False
        )
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, tokens, targets=None):
        B, T = tokens.shape
        
        # Token embeddings
        x = self.tok_emb(tokens)  # (B, T, dim)
        
        # Get RoPE frequencies for this sequence length
        freqs_cis = self.freqs_cis[:T].to(x.device)
        
        # Pass through transformer blocks
        for layer in self.layers:
            x = layer(x, freqs_cis)
        
        # Final norm + project to vocab
        x = self.norm(x)
        logits = self.output(x)  # (B, T, vocab_size)
        
        # Compute loss if targets provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, tokens, max_new_tokens, temperature=0.8, top_k=50):
        """Generate text autoregressively."""
        for _ in range(max_new_tokens):
            # Crop to max_seq_len if needed
            ctx = tokens[:, -self.max_seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tokens = torch.cat([tokens, next_token], dim=1)
        
        return tokens

## 5. Model Configuration

Three sizes tuned for 3080 Ti (12GB VRAM):

| Config | Params | Layers | Dim | Heads | KV Heads | VRAM |
|--------|--------|--------|-----|-------|----------|------|
| Small  | ~15M   | 6      | 384 | 6     | 2        | ~2 GB |
| Medium | ~45M   | 8      | 512 | 8     | 2        | ~4 GB |
| Large  | ~110M  | 12     | 768 | 12    | 4        | ~8 GB |

In [ ]:
# === CHOOSE YOUR CONFIG ===
# Start with 'small' to learn, then scale up to 'large' for better results

CONFIGS = {
    'small': dict(dim=384, n_layers=6, n_heads=6, n_kv_heads=2),
    'medium': dict(dim=512, n_layers=8, n_heads=8, n_kv_heads=2),
    'large': dict(dim=768, n_layers=12, n_heads=12, n_kv_heads=4),
}

config_name = 'small'  # <-- Change this to scale up
cfg = CONFIGS[config_name]

# Training hyperparameters
BATCH_SIZE = 64
SEQ_LEN = 256
MAX_ITERS = 5000
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
EVAL_INTERVAL = 250
EVAL_ITERS = 50

# Create model
model = MiniLLaMA(
    vocab_size=vocab_size,
    max_seq_len=SEQ_LEN,
    **cfg
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f'Config: {config_name}')
print(f'Parameters: {n_params:,} ({n_params/1e6:.1f}M)')
print(f'\nModel architecture:\n{model}')

## 6. Training Loop

Key training techniques:
- **AdamW optimizer** — Adam with decoupled weight decay (standard for transformers)
- **Cosine learning rate schedule** — gradually reduces LR for better convergence
- **Mixed precision (AMP)** — uses float16 where safe, saves VRAM and speeds up training
- **Gradient clipping** — prevents exploding gradients

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=EVAL_ITERS):
    """Estimate loss on train and val sets."""
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        batch_losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            xb, yb = get_batch(split, BATCH_SIZE, SEQ_LEN)
            with autocast('cuda', dtype=torch.float16):
                _, loss = model(xb, yb)
            batch_losses[i] = loss.item()
        losses[split] = batch_losses.mean().item()
    model.train()
    return losses

In [ ]:
# Optimizer with weight decay (don't decay embeddings, norms, biases)
decay_params = [p for n, p in model.named_parameters() if p.dim() >= 2]
nodecay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': WEIGHT_DECAY},
    {'params': nodecay_params, 'weight_decay': 0.0},
], lr=LEARNING_RATE, betas=(0.9, 0.95))

# Cosine LR schedule with warmup
warmup_iters = 100
def get_lr(it):
    if it < warmup_iters:
        return LEARNING_RATE * it / warmup_iters
    if it > MAX_ITERS:
        return LEARNING_RATE * 0.1
    decay_ratio = (it - warmup_iters) / (MAX_ITERS - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return LEARNING_RATE * 0.1 + coeff * (LEARNING_RATE - LEARNING_RATE * 0.1)

# Mixed precision scaler
scaler = GradScaler('cuda')

# Training tracking
train_losses = []
val_losses = []
lrs = []

print(f'Starting training for {MAX_ITERS} iterations...')
print(f'Batch size: {BATCH_SIZE} | Seq length: {SEQ_LEN}')
print(f'Tokens per iteration: {BATCH_SIZE * SEQ_LEN:,}')
print(f'Total tokens seen: ~{BATCH_SIZE * SEQ_LEN * MAX_ITERS / 1e6:.1f}M')
print('-' * 60)

In [ ]:
# === TRAINING LOOP ===
model.train()
pbar = tqdm(range(MAX_ITERS), desc='Training')

for step in pbar:
    # Update learning rate
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    lrs.append(lr)
    
    # Get batch
    xb, yb = get_batch('train', BATCH_SIZE, SEQ_LEN)
    
    # Forward pass with mixed precision
    with autocast('cuda', dtype=torch.float16):
        logits, loss = model(xb, yb)
    
    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    
    # Update progress bar
    pbar.set_postfix({'loss': f'{loss.item():.3f}', 'lr': f'{lr:.1e}'})
    
    # Evaluate periodically
    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS - 1:
        losses = estimate_loss(model)
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        print(f"\nStep {step:5d} | Train loss: {losses['train']:.4f} | Val loss: {losses['val']:.4f}")
        
        # Generate a sample
        prompt = torch.tensor([encode('\n')], device=device)
        sample = model.generate(prompt, max_new_tokens=100, temperature=0.8)
        print(f"Sample: {decode(sample[0].tolist())[:150]}")
        print('-' * 60)
        model.train()

print('\nTraining complete!')

## 7. Training Visualization

Let's plot the training curves to understand how the model learned.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curves
steps = [i * EVAL_INTERVAL for i in range(len(train_losses))]
axes[0].plot(steps, train_losses, label='Train', linewidth=2)
axes[0].plot(steps, val_losses, label='Val', linewidth=2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate schedule
axes[1].plot(lrs, linewidth=2, color='green')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Cosine LR Schedule with Warmup')
axes[1].grid(True, alpha=0.3)

# Perplexity
train_ppl = [math.exp(l) for l in train_losses]
val_ppl = [math.exp(l) for l in val_losses]
axes[2].plot(steps, train_ppl, label='Train', linewidth=2)
axes[2].plot(steps, val_ppl, label='Val', linewidth=2)
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Perplexity')
axes[2].set_title('Perplexity (lower = better)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 8. Save & Load Model

In [ ]:
os.makedirs('checkpoints', exist_ok=True)
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': cfg,
    'config_name': config_name,
    'vocab_size': vocab_size,
    'seq_len': SEQ_LEN,
    'train_losses': train_losses,
    'val_losses': val_losses,
    'chars': chars,
}
path = f'checkpoints/mini_llama_{config_name}.pt'
torch.save(checkpoint, path)
size_mb = os.path.getsize(path) / 1e6
print(f'Saved checkpoint to {path} ({size_mb:.1f} MB)')

## 9. Interactive Text Generation Playground

Try generating text with different settings!

In [ ]:
def generate_text(prompt_text, max_tokens=500, temperature=0.8, top_k=50):
    """Generate text from a prompt."""
    model.eval()
    tokens = torch.tensor([encode(prompt_text)], device=device)
    output = model.generate(tokens, max_new_tokens=max_tokens, 
                           temperature=temperature, top_k=top_k)
    return decode(output[0].tolist())

# Try different prompts and temperatures!
print('=' * 60)
print('GENERATION: temperature=0.8 (balanced)')
print('=' * 60)
print(generate_text('ROMEO:', max_tokens=300, temperature=0.8))

print('\n' + '=' * 60)
print('GENERATION: temperature=0.3 (more focused)')
print('=' * 60)
print(generate_text('To be, or not to be', max_tokens=300, temperature=0.3))

print('\n' + '=' * 60)
print('GENERATION: temperature=1.2 (more creative)')
print('=' * 60)
print(generate_text('The king', max_tokens=300, temperature=1.2))

## 10. Model Analysis: Attention Visualization

Let's look at what the attention heads have learned.

In [ ]:
def visualize_attention(text_input, layer_idx=0):
    """Visualize attention patterns for a given input."""
    model.eval()
    tokens = torch.tensor([encode(text_input)], device=device)
    T = tokens.shape[1]
    
    # Hook to capture attention weights
    attn_weights = []
    def hook_fn(module, input, output):
        # Re-derive attention weights from Q, K
        x = input[0]  # input to the attention module's parent
        return output
    
    # Manual forward through the target layer to get Q, K
    with torch.no_grad():
        x = model.tok_emb(tokens)
        freqs_cis = model.freqs_cis[:T].to(x.device)
        
        for i, layer in enumerate(model.layers):
            if i == layer_idx:
                # Get attention internals
                normed = layer.norm1(x)
                attn = layer.attention
                B, T_len, C = normed.shape
                q = attn.wq(normed).view(B, T_len, attn.n_heads, attn.head_dim)
                k = attn.wk(normed).view(B, T_len, attn.n_kv_heads, attn.head_dim)
                q = apply_rope(q, freqs_cis)
                k = apply_rope(k, freqs_cis)
                
                if attn.n_rep > 1:
                    k = k.unsqueeze(3).expand(B, T_len, attn.n_kv_heads, attn.n_rep, attn.head_dim)
                    k = k.reshape(B, T_len, attn.n_heads, attn.head_dim)
                
                q = q.transpose(1, 2)
                k = k.transpose(1, 2)
                scores = (q @ k.transpose(-2, -1)) / math.sqrt(attn.head_dim)
                # Apply causal mask
                mask = torch.triu(torch.ones(T_len, T_len, device=device), diagonal=1).bool()
                scores.masked_fill_(mask, float('-inf'))
                attn_map = F.softmax(scores, dim=-1)
                break
            x = layer(x, freqs_cis)
    
    # Plot first 4 heads
    n_show = min(4, attn_map.shape[1])
    fig, axes = plt.subplots(1, n_show, figsize=(4*n_show, 4))
    if n_show == 1:
        axes = [axes]
    
    chars_list = list(text_input[:T_len])
    for h in range(n_show):
        im = axes[h].imshow(attn_map[0, h].cpu().numpy(), cmap='viridis')
        axes[h].set_title(f'Head {h}')
        if len(chars_list) <= 30:
            axes[h].set_xticks(range(len(chars_list)))
            axes[h].set_yticks(range(len(chars_list)))
            axes[h].set_xticklabels(chars_list, rotation=90, fontsize=7)
            axes[h].set_yticklabels(chars_list, fontsize=7)
    
    plt.suptitle(f'Attention Patterns — Layer {layer_idx}', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_attention('To be or not to be', layer_idx=0)
visualize_attention('To be or not to be', layer_idx=max(0, cfg['n_layers']-1))

## 11. Exercises to Deepen Your Understanding

Complete these to strengthen your portfolio:

### Exercise 1: Scale Up
Change `config_name` to `'medium'` or `'large'` and retrain. Compare loss curves.

### Exercise 2: Hyperparameter Search
Try different learning rates (1e-3, 1e-4), batch sizes (32, 128), and sequence lengths.

### Exercise 3: Replace GQA with Standard MHA
Set `n_kv_heads = n_heads` and compare speed and quality.

### Exercise 4: Add Dropout
Add dropout to attention and FFN. Does it help with the train/val gap?

### Exercise 5: Train on Your Own Data
Replace Shakespeare with any text file. Just change `data_path` and re-run.

## 12. Key Concepts Summary (for Interviews)

**Q: Why RMSNorm instead of LayerNorm?**
A: RMSNorm skips the mean subtraction step — only normalizes by root-mean-square. This is ~10-15% faster with no quality loss. Used in LLaMA, Mistral, Gemma.

**Q: What is RoPE and why is it better than learned positional embeddings?**
A: RoPE encodes position by rotating Q and K vectors. The dot product depends on *relative* position, enabling better length generalization. No extra parameters needed.

**Q: What is SwiGLU?**
A: A gated FFN variant: `Swish(W1·x) * W2·x`. The gating mechanism lets the model learn which features to amplify. Consistently outperforms ReLU/GELU.

**Q: Explain Grouped Query Attention.**
A: GQA shares K and V projections across groups of Q heads. This reduces KV cache size (critical for inference) with minimal quality loss. LLaMA 3 uses 8 KV heads for 32 Q heads.

**Q: Why weight tying between embedding and output?**
A: The embedding maps tokens → vectors. The output maps vectors → token probabilities. These are conceptually inverse operations, so sharing weights reduces parameters and acts as regularization.

**Q: Why AdamW over Adam?**
A: AdamW decouples weight decay from the gradient update. In standard Adam, weight decay interacts with the adaptive learning rate; AdamW applies it directly to weights, which is more principled.

**Q: Why cosine LR schedule?**
A: Cosine smoothly decreases the learning rate, allowing large updates early (fast learning) and fine-grained updates late (convergence). Warmup prevents instability at the start.